## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# Load variables from .env
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize model
model = init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B1AF9E9550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B1AF9EA270>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="This title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B1AF9E9550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B1AF9EA270>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [7]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Inception is a movie directed by Christopher Nolan, right? It\'s a sci-fi action film. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a thief who steals secrets by entering the subconscious of his targets. The film\'s plot revolves around the concept of planting an idea into someone\'s mind, which is called "inception." \n\nThe movie\'s visual effects are pretty famous, especially the rotating hallway fight scene. The cast includes some big names like Joseph Gordon-Levitt, who plays Arthur, a team member. Ellen Page is there as Ariadne, a young architect who helps design the dream environments. There\'s also Tom Hardy as Eames, who can mimic other people\'s appearances. And Ken Watanabe as Saito, a Japanese business tycoon. \n\nThe story structure is a bit complex, with multiple layers of dreams, each one deeper than the last. The

In [8]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message Output Alongside Parsed Structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details. """
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a Movie function that requires title, year, director, and rating. I need to fill those in. I remember Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and director. Yep, 2010 and Christopher Nolan. The rating is 8.8. So I'll structure the tool call with those parameters.\n", 'tool_calls': [{'id': '9wp77wkwe', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 232, 'total_tokens': 395, 'completion_time': 0.262356158, 'completion_tokens_details': {'reasoning_tokens': 115}, 'prompt_time': 0.010067407, 'prompt_tokens_details': None, 'queue_time': 0.31805999

### Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model__with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There's a Movie function that requires title, year, director, and rating. I need to fill those parameters. I know Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and rating. Yep, 2010 and 8.8. So I'll structure the tool call with those details.\n", 'tool_calls': [{'id': 'khc8h821z', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 159, 'prompt_tokens': 232, 'total_tokens': 391, 'completion_time': 0.275688996, 'completion_tokens_details': {'reasoning_tokens': 111}, 'prompt_time': 0.011917162, 'prompt_tokens_details': None, 'queue_time': 0.282878344, 'total_time': 0.287606

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [12]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")

response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [16]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [17]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [18]:
import os

os.environ["NVIDIA_API_KEY"] = os.getenv("NVIDIA_API_KEY")

In [21]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "meta/llama-3.1-70b-instruct",
    model_provider="nvidia"
)

response = model.invoke("Hello")
print(response.content)

Hello. What can I help you with?


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="nvidia:meta/llama-3.1-70b-instruct",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{   
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, +91 9999321234"
    }]
})

print(result["structured_response"])

name='extract_contact_info' email='john@example.com' phone='+91 9999321234'


In [25]:
result["structured_response"]

ContactInfo(name='extract_contact_info', email='john@example.com', phone='+91 9999321234')

In [28]:
## TypeDict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str 
    email: str
    phone: str

agent = create_agent(
    model="nvidia:meta/llama-3.1-70b-instruct",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{   
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, +91 9999321234"
    }]
})

print(result["structured_response"])

{'name': 'extract_contact_info', 'email': 'john@example.com', 'phone': '+91 9999321234'}


In [29]:
# Dataclass 

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass

class ContactInfo:
    """Contact information for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model="nvidia:meta/llama-3.1-70b-instruct",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{   
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, +91 9999321234"
    }]
})

print(result["structured_response"])

ContactInfo(name='extract_contact_info', email='john@example.com', phone='+91 9999321234')
